# PI — 예측 분위수와 통합성적표 만들기

분기 시계열 CSV 한 개와 첫 origin 날짜를 넣으면 모델을 자동으로 고르고, 모든 origin의 **예측 분위수(딕셔너리 1)** 와 **통합성적표(딕셔너리 2)** 를 만들어 줍니다. 이 폴더만 있으면 실행됩니다.


## 폴더와 파일

```
PI_module/
├─ PI.py          실행 파일(명령줄과 함수 겸용)  ← 사용자는 PI.py 또는 PI.ipynb 중 하나만 실행
├─ PI.ipynb       같은 일을 셀 단위로 (▶ 작동 코드 셀만 실행해도 결과가 나옴, ▷ 설명 코드 셀은 확인용)
├─ selection.py   모든 조합 백테스트와 모델 선정 (model_selection/ 복사본, 직접 실행하지 않음)
├─ ts_models.py   AR 단독, GARCH, SV 모델 계산 (model_selection/ 복사본, 직접 실행하지 않음)
├─ test_PI.py     테스트
├─ data/          입력 CSV
├─ cache/         계산 캐시 (실행하면 생김)
└─ output/        결과
```

- `PI.py`와 `PI.ipynb`는 같은 폴더에 같은 이름의 결과를 저장하므로, 같은 입력으로 나중에 실행한 쪽이 덮어씁니다.


## 빠른 시작

```
python PI.py <CSV 이름> <첫 origin의 마지막 분기 날짜>
python PI.py X1_l1 '2019-12-01'
```

- 노트북으로 하려면 `PI.ipynb`에서 `csv_name`, `first_insample_end_date` 두 셀만 바꾸고 위에서 아래로 실행합니다.
- CSV는 이름만 주면 현재 폴더, `data/` 순으로 찾습니다(`.csv` 생략 가능).
- 첫 실행은 변수 하나에 2~5분 걸립니다. 같은 입력으로 다시 실행하면 캐시(`cache/`)를 읽어 바로 끝납니다.
- 파이썬에서는 `from PI import run_pi; d1, d2 = run_pi("X1_l1", "2019-12-01")`.
- 필요한 패키지: numpy, pandas, scipy, statsmodels, arch, pmdarima, properscoring (`pip install -r requirements.txt` 또는 `uv sync`).


## 입력

**CSV**: `Date`, `Data` 두 열, 연속된 분기 자료.

```
Date,Data
2011-06-01,70.10
2011-09-01,-220.88
```

- `Date`는 그 날짜가 속한 분기로 읽습니다(2018-12-01 → 2018-Q4).
- 최소 28분기가 필요합니다. 맨 앞뒤의 빈 행은 자동으로 잘라 냅니다.
- 값이 모두 양수일 때만 로그 스케일도 후보가 됩니다.

**첫 origin 날짜**: 이 날짜가 속한 분기까지를 훈련 데이터로 쓰는 것이 첫 origin이고, 이후 데이터 끝까지 한 분기씩 origin이 이어집니다.

- 가장 이른 분기는 데이터의 24번째 분기, 가장 늦은 분기는 끝에서 5번째 분기입니다.
- 이른 날짜는 보정 구간이 짧아 구간이 불안정합니다. 첫 origin이 훈련 30분기 이후(보정 10개 이상)인 것을 권합니다.

**입력에 문제가 있으면** 계산 전에 원인과 위치를 알려 주는 오류로 멈춥니다. 메시지에 나온 대로 고치면 됩니다.

| 문제 | 메시지의 핵심 | 고치는 법 |
|---|---|---|
| 파일·열 | `CSV를 찾지 못했다`, `Date, Data 두 열이 있어야 한다` | 이름과 열 이름 확인 |
| 값 | 숫자가 아닌 값, 가운데 빈 값, 모두 같은 값, 28분기 미만 | 표시된 날짜의 값을 고치거나 채우기 |
| 날짜 | 날짜로 읽지 못함, 분기가 연속이 아님 | 빠진 분기는 행과 값을 채우고, 중복·순서를 정리 |
| 첫 origin | 형식 오류, 데이터 범위 밖, 너무 이르거나 늦음 | 메시지에 나온 가능한 분기로 바꾸기 |


## 출력

`output/<이름>/`에 변수마다 폴더가 생기고 파일 4개가 저장됩니다.

| 파일 | 내용 |
|---|---|
| `<이름>_forecast.json` | 딕셔너리 1. `json.load`로 읽으면 딕셔너리가 그대로 나옵니다. |
| `<이름>_score.json` | 딕셔너리 2 |
| `<이름>_forecast.csv` | 딕셔너리 1을 펼친 표. 열: `origin, h, 예측 분기, p5 ~ p95` |
| `<이름>_score.csv` | 딕셔너리 2 |

**딕셔너리 1** — 예측 분위수

```python
d1["2018-Q4"]    # 9 x 4 배열
# 행: 5, 10, 20, 30, 50, 70, 80, 90, 95% 분위수 / 열: h1 ~ h4 (원래 단위)
```

- **key는 훈련에 쓴 마지막 분기**(origin)이고, 열은 **그다음 4분기**입니다. 위 예에서 열 h1~h4는 2019-Q1, Q2, Q3, Q4의 예측이며 2018-Q4 자체는 들어 있지 않습니다.
- 첫 origin부터 데이터 끝까지 모든 origin이 들어 있습니다. 마지막 origin은 데이터 끝 이후 4분기를 예측한 값입니다(정답 없음).
- CSV의 `예측 분기`는 `2019Q1`처럼 하이픈 없이 쓰고, key는 `2018-Q4`처럼 하이픈이 있습니다.

**딕셔너리 2** — 통합성적표 `{'mse', 'crps', 'coverage'}`

- 4분기 정답이 모두 있는 origin의 평균입니다.
- `mse`, `crps`는 전체 표준편차로 정규화한 값이고, `coverage`는 90% 구간(5~95%) 적중률(%)입니다.

화면 출력 예:

```
선정 모델: GARCH (d=0, D=0) 원 스케일, 리샘플링 구간 | 후보 6/24개, 보정 origin 15개
딕셔너리 2: {'mse': 1.5448, 'crps': 0.7453, 'coverage': 82.1429}
```


In [1]:
import json
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
NAME = "X1_l1"                      # PI_module/output/ 안의 예시 결과(X1)
d1 = json.load(open(f"output/{NAME}/{NAME}_forecast.json", encoding="utf-8"))
d2 = json.load(open(f"output/{NAME}/{NAME}_score.json", encoding="utf-8"))
print("딕셔너리 2 (통합성적표):", {k: round(v, 4) for k, v in d2.items()})
key = "2019-Q4"
print(f"\n딕셔너리 1[{key!r}]  -  {key}까지의 데이터로 만든 예측 (행: 분위수, 열: 몇 분기 앞)")
display(pd.DataFrame(d1[key], index=["5%", "10%", "20%", "30%", "50%", "70%", "80%", "90%", "95%"], columns=["h1", "h2", "h3", "h4"]).round(1))
f = pd.read_csv(f"output/{NAME}/{NAME}_forecast.csv")
print("\n_forecast.csv (긴 표) 앞 8행")
display(f.head(8).round(1))
g = f[f["h"] == 1]
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.fill_between(g["예측 분기"], g["p5"], g["p95"], color="#9ec5e8", alpha=.5, label="5~95% 구간")
ax.fill_between(g["예측 분기"], g["p20"], g["p80"], color="#4a90d9", alpha=.35, label="20~80% 구간")
ax.plot(g["예측 분기"], g["p50"], color="#1f4e8c", label="50%")
ax.set_title("h=1 예측 구간 (origin마다 한 분기 앞)", fontweight="bold")
ax.tick_params(axis="x", rotation=60, labelsize=8)
ax.legend(frameon=False, ncol=3)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
plt.tight_layout(); plt.show()

딕셔너리 2 (통합성적표): {'mse': 1.5448, 'crps': 0.7453, 'coverage': 82.1429}

딕셔너리 1['2019-Q4']  -  2019-Q4까지의 데이터로 만든 예측 (행: 분위수, 열: 몇 분기 앞)


,h1,h2,h3,h4
5%,-151.5,-192.4,-143.0,-150.0
10%,-116.7,-117.9,-134.8,-150.0
20%,17.5,-21.0,-71.2,-79.8
30%,55.1,-0.7,-66.1,-77.4
50%,85.8,49.6,9.6,-5.3
70%,179.9,118.4,112.2,45.6
80%,218.7,211.9,166.5,85.9
90%,297.9,272.2,171.2,147.5
95%,515.0,379.6,243.3,251.6



_forecast.csv (긴 표) 앞 8행


,origin,h,예측 분기,p5,p10,p20,p30,p50,p70,p80,p90,p95
0,2019-Q4,1,2020Q1,-151.5,-116.7,17.5,55.1,85.8,179.9,218.7,297.9,515.0
1,2019-Q4,2,2020Q2,-192.4,-117.9,-21.0,-0.7,49.6,118.4,211.9,272.2,379.6
2,2019-Q4,3,2020Q3,-143.0,-134.8,-71.2,-66.1,9.6,112.2,166.5,171.2,243.3
3,2019-Q4,4,2020Q4,-150.0,-150.0,-79.8,-77.4,-5.3,45.6,85.9,147.5,251.6
4,2020-Q1,1,2020Q2,-263.8,-259.1,-123.5,-73.9,-22.4,104.0,146.5,233.2,470.9
5,2020-Q1,2,2020Q3,-146.2,-142.4,-123.0,-45.4,-10.7,94.2,102.3,248.3,355.9
6,2020-Q1,3,2020Q4,-153.3,-145.2,-121.5,-82.6,-18.2,41.0,151.2,155.8,226.7
7,2020-Q1,4,2021Q1,-152.3,-113.6,-105.1,-83.6,-10.7,37.8,78.6,138.9,240.8


<Figure size 1100x360 with 1 Axes>

## 작동 방식

```
CSV ─▶ 입력 검사 ─▶ 모든 조합 백테스트 ─▶ 모델 선정 ─▶ 선정 조합으로 분위수 계산 ─▶ 딕셔너리 1·2 저장
```

1. **백테스트**: 모델 3종(AR 단독, GARCH(1,1), SV) × 차분 (d, D) 4가지 × 스케일(원, 로그) × 구간 방식(자체, 리샘플링)을 origin마다 한 분기씩 늘려 가며 적합하고 h=1~4를 예측합니다. 훈련 구간만 쓰므로 미래 정보는 들어가지 않습니다.
2. **선정**: CRPS 1등의 ±10% 이내 후보 중 커버리지가 90%에 가장 가까운 조합을 고릅니다.
3. **분위수**: 선정 조합 하나를 같은 시드(42)로 다시 계산해 origin마다 9개 분위수를 만듭니다. 같은 입력이면 같은 결과가 나옵니다.
4. **캐시**: 1번의 결과를 `cache/`에 저장합니다. CSV에 새 분기를 덧붙이면 바뀐 부분 뒤쪽만 다시 계산하고, 첫 origin 날짜를 바꾸면 새로 계산합니다.


## 테스트

```
python test_PI.py          # 빠른 테스트 28개 (캐시를 써서 금방 끝남)
python test_PI.py full     # + 변수별 날짜 8개, 새로 계산해서 30분 안팎 (총 36개)
```

마지막 줄에 `실패 0개`가 나오면 정상입니다. 평소에는 빠른 테스트면 충분하고, `full`은 모델 계산 코드를 크게 고쳤을 때만 돌립니다.


## 알아 둘 점

- 모델 선정과 성적 평가가 같은 구간(첫 origin 이후)이라 점수가 다소 낙관적일 수 있습니다.
- horizon은 h=1~4로 고정입니다.
